# Per-residue L¹ dihedral distance vs PTI-PAE

Generates the scatter plot of mean per-residue CDR dihedral distance ($\bar d_{L1} = \langle |\Delta\varphi| + |\Delta\psi|\rangle$) against AF3 PTI-PAE for the 16 class I validation triads.

**Input**: `ramachandran_all_classI.csv` — per-residue CDR φ/ψ for predicted and crystal, one row per residue, columns `pdb`, `pti_pae`, `phi_pred`, `psi_pred`, `phi_crystal`, `psi_crystal`. Produced by `extract_rama_all.py`.

**Output**: `dihed_L1_vs_pae_fullrange.png` next to this notebook.


In [ ]:
# --- Config ---
INPUT_CSV  = 'ramachandran_all_classI.csv'      # edit if path differs
OUTPUT_PNG = 'dihed_L1_vs_pae_fullrange.png'
CLASS_COLOR = '#cc4444'   # class I red
DPI = 170

import pandas as pd
import numpy as np
from scipy.stats import pearsonr, spearmanr
import matplotlib.pyplot as plt

In [ ]:
# --- Load per-residue data ---
df = pd.read_csv(INPUT_CSV)
print(f'Loaded {len(df)} residues across {df["pdb"].nunique()} triads')
df.head()

In [ ]:
# --- Compute per-residue L¹ dihedral distance (circular Δφ + circular Δψ) ---
def circ_deg(a, b):
    """Smallest angular distance between two angles in degrees, in [0, 180]."""
    return np.abs((a - b + 180.0) % 360.0 - 180.0)

df['dphi'] = circ_deg(df['phi_pred'], df['phi_crystal'])
df['dpsi'] = circ_deg(df['psi_pred'], df['psi_crystal'])
df['d_L1'] = df['dphi'] + df['dpsi']    # L¹ city-block; max possible = 360°

# Per-triad average
per_triad = df.groupby('pdb').agg(
    pti_pae=('pti_pae', 'first'),
    dihed_L1=('d_L1', 'mean'),
    n_residues=('d_L1', 'size'),
).reset_index().sort_values('pti_pae')
per_triad

In [ ]:
# --- Correlation stats ---
r_p, p_p = pearsonr(per_triad['pti_pae'], per_triad['dihed_L1'])
r_s, p_s = spearmanr(per_triad['pti_pae'], per_triad['dihed_L1'])
print(f'Pearson  r = {r_p:.3f}   p = {p_p:.3f}')
print(f'Spearman ρ = {r_s:.3f}   p = {p_s:.3f}')

In [ ]:
# --- Plot: y-axis runs the full theoretical range [0, 360°] ---
fig, ax = plt.subplots(figsize=(7.5, 6.5))
ax.scatter(per_triad['pti_pae'], per_triad['dihed_L1'],
           s=110, edgecolor='black', linewidth=0.7,
           color=CLASS_COLOR, zorder=3, label='Class I')
for _, row in per_triad.iterrows():
    ax.annotate(row['pdb'], (row['pti_pae'], row['dihed_L1']),
                xytext=(6, 4), textcoords='offset points',
                fontsize=9, alpha=0.85)

# Theoretical-max reference line
ax.axhline(360, color='gray', lw=0.5, ls=':', alpha=0.7)
ax.text(ax.get_xlim()[1] * 0.98, 358,
        'theoretical max (|Δφ| + |Δψ| ≤ 360°)',
        ha='right', va='top', fontsize=8, color='gray')

ax.set_ylim(0, 365)
ax.set_xlabel('AF3 PTI-PAE')
ax.set_ylabel(r'Mean per-residue CDR dihedral distance, L$^{1}$  '
              r'$\langle|\Delta\varphi| + |\Delta\psi|\rangle$  (°)')
ax.set_title(
    f'Per-residue L$^1$ dihedral distance vs PTI-PAE — '
    f'{len(per_triad)} class I validation triads  '
    f'(Pearson r = {r_p:.2f}, p = {p_p:.2f})')
ax.grid(True, alpha=0.3)
ax.legend(frameon=False, loc='upper left')
plt.tight_layout()
plt.savefig(OUTPUT_PNG, dpi=DPI, bbox_inches='tight')
plt.show()
print(f'saved -> {OUTPUT_PNG}')